# Logistic Regression (Simple Example)

Predict the **probability** of a binary outcome (spam vs. ham) using the **sigmoid**:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = m x + b$$

We fit `m` and `b` by mapping labels to compressed log-odds and using plain least squares on the logit scale:

$$z = \ln\!\left(\frac{p}{1-p}\right) = m x + b$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dataset: 6 emails. X = trigger-word count, Y = label (1 = spam, 0 = ham)
X = np.array([2, 4, 8, 12, 14, 18], dtype=float)
Y = np.array([0, 0, 1, 1, 1, 1], dtype=float)

print(pd.DataFrame({"X (trigger words)": X.astype(int), "label": Y.astype(int)}).to_string(index=False))


## Fit m and b (logit-linearization + OLS)

Compress labels to `0.08` / `0.92` (avoids `log(0)`), convert to log-odds, then fit a line with the normal OLS formulas.


In [ ]:
Y_smooth = np.where(Y == 1, 0.92, 0.08)
logits = np.log(Y_smooth / (1 - Y_smooth))

x_mean = np.mean(X)
l_mean = np.mean(logits)

numerator = np.sum((X - x_mean) * (logits - l_mean))
denominator = np.sum((X - x_mean) ** 2)
m = numerator / denominator
b = l_mean - m * x_mean

print("=== LEARNED PARAMETERS ===")
print(f"Weight (m) : {m:.4f}")
print(f"Bias (b)   : {b:.4f}")


## Predictions, Decision Boundary & Accuracy


In [ ]:
probs = 1.0 / (1.0 + np.exp(-(m * X + b)))
preds = (probs >= 0.5).astype(int)
accuracy = np.mean(preds == Y)
boundary_x = -b / m

print("=== FINAL PREDICTIONS ===")
pred_table = pd.DataFrame({
    "X (trigger words)": X.astype(int),
    "True label": Y.astype(int),
    "P(spam)": np.round(probs, 4),
    "Predicted": preds.astype(int),
    "Correct": (preds == Y).astype(int),
})
print(pred_table.to_string(index=False))
print()
print(f"Decision boundary x* = -b/m = {boundary_x:.3f} (P = 0.5)")
print(f"Training accuracy    : {accuracy*100:.1f}%")


## Plot


In [ ]:
x_line = np.linspace(0, 20, 200)
p_line = 1.0 / (1.0 + np.exp(-(m * x_line + b)))

plt.figure(figsize=(7, 4.5))
plt.scatter(X, Y, color="crimson", s=80, zorder=5, label="Emails (1=spam, 0=ham)")
plt.plot(x_line, p_line, color="teal", linewidth=2.5, label="Logistic curve")
plt.axvline(boundary_x, color="darkorange", linestyle="-.", linewidth=1.5,
            label=f"Decision cutoff ({boundary_x:.2f})")
plt.axhline(0.5, color="gray", linestyle=":", linewidth=1)
plt.title("Logistic Regression: Spam Probability")
plt.xlabel("Trigger Word Count")
plt.ylabel("P(spam)")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
